# River-basin runoff onset

HydroBASINS level-5 basins: pixel-weighted median onset and MAD as basin
choropleths, annual anomalies by basin, regional views (High Mountain Asia, western US) with
population and — if `snow_water.ipynb` has been run — the share of precipitation falling as
snow. Reads the river-basin cube written by `pipeline/scripts/reduce_partials.py`.

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import seaborn as sns
import xarray as xr
from cartopy import crs as ccrs
from cartopy import feature as cfeature

from gsro_analysis import aggregate, paths, settings, stats

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

# the river-basin cube: basin x elevation x chili_class x water_year (+ ERA5 anomaly zonal means)
basins_ds = aggregate.open_aggregate('river_basins', config.version)
basins_ds

In [ ]:
# HydroBASINS level 5 from the locally cached BasinATLAS gdb (the pipeline stores level-6 ids and
# the default river_basins cube is their level-5 prefix, so these are the cube's polygons; the gdb
# spells the reserved word ORDER with a trailing underscore)
basins_gdf = gpd.read_file(
    settings.cached_source(settings.BASIN_ATLAS_URL, filename='BasinATLAS_Data_v10.gdb.zip',
                           expected_md5=settings.BASIN_ATLAS_MD5),
    layer=settings.basin_atlas_layer(5)).rename(columns={'ORDER_': 'ORDER'})
basin_populations_gdf = gpd.read_file(paths.GEOMETRIES / 'Hydrobasins_L5_Population_Global.geojson')
print('total population in all basins (billion):', basin_populations_gdf['total_population'].sum() / 1e9)

# one row per basin: geometry, population, pixel-weighted means of median onset / MAD / yearly
# onset and anomaly, pixel counts and the mapped share of the basin area (means masked where
# < 5 % of the basin is mapped, yearly values where < 1 %)
basins_means_gdf = stats.basin_summary(basins_ds, basins_gdf, basin_populations_gdf)
basins_means_gdf

## Global maps

In [ ]:
# row-strip chunks: the GeoTIFF is striped, so 'auto' 2-D chunks make every dask
# thread decode the whole raster (5.5 GB in seconds on 2026-08-25)
hillshade_da = rxr.open_rasterio(paths.DATA / 'global_hillshade_robinson.tif', masked=True, chunks={'y': 2048, 'x': -1}).squeeze().coarsen(x=10,y=10, boundary='trim').mean().compute()
hillshade_da

In [ ]:
basins_means_robinson_gdf = basins_means_gdf.to_crs("ESRI:54030")

In [ ]:
f,axs = plt.subplots(2,1,figsize=(10,8), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

basins_means_robinson_gdf.plot(ax=axs[0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=ccrs.Robinson(),vmin=110,vmax=250)
basins_means_robinson_gdf.plot(ax=axs[1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=ccrs.Robinson(),vmin=0,vmax=30)

for ax in axs:
    
    hillshade_da.plot.imshow(ax=ax,cmap='gray', transform=ccrs.Robinson(), zorder=0,add_colorbar=False)
    gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-90, -60, -30, 0, 30, 60, 90], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True
    ax.set_title("")
    #ax.set_global()

    #ax.set_extent([-180, 180, -63, 90], crs=ccrs.PlateCarree())

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
    
axs[0].set_title("10-year Median Snowmelt Runoff Onset by River Basin")
axs[1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin")

f.savefig(paths.figdir('river_basins', config.version) / 'global_median_runoff_onset_and_mad.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
f,ax=plt.subplots(nrows=5,ncols=2,figsize=(10,12),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(basins_ds.water_year.values, ax.flat):
    basins_means_robinson_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=ccrs.Robinson())
    hillshade_da.plot.imshow(ax=ax,cmap='gray', transform=ccrs.Robinson(), zorder=0,add_colorbar=False)
    ax.set_title(f'WY{water_year}')

    #ax.set_global()
    #gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-90, -60, -30, 0, 30, 60, 90], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True

    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

f.savefig(paths.figdir('river_basins', config.version) / 'global_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
#f.suptitle('Runoff Onset Anomaly by River Basin')
# f.tight_layout()

In [ ]:
f,axes=plt.subplots(nrows=2,ncols=5,figsize=(15,6.2),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(basins_ds.water_year.values, axes.flat):
    basins_means_robinson_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=ccrs.Robinson())
    ax.set_title(f'WY{water_year}')
    gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, linestyle='--', linewidth=0.5)
    #gl.top_labels=False
    # gl.bottom_labels=True
    #gl.right_labels=False
    # gl.left_labels=True

    # turn on left labels for WY2015 and WY2020 only, turn on bottom labels for WY 2020-WY2024
    # if water_year == 2015 or water_year == 2020:
    #     gl.left_labels = True
    # else:
    #     gl.left_labels = False
    # if water_year in [2020, 2021, 2022, 2023, 2024]:
    #     gl.bottom_labels = True
    # else:
    #     gl.bottom_labels = False


# Add a single colorbar spanning both rows
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=-30, vmax=30)
sm = plt.cm.ScalarMappable(cmap='RdBu', norm=norm)
sm.set_array([])
cbar = f.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8, aspect=20, pad=0.02, extend='both')
cbar.set_label('Snowmelt runoff onset anomaly [days]')

## Regional views — run a whole block (`hma` or `wus`) top to bottom

In [ ]:
region = 'hma'
#region = 'wus'

if region == 'hma':
    cartopy_aea_crs = ccrs.AlbersEqualArea(central_latitude=32.5, central_longitude=90)
if region == 'wus':
    cartopy_aea_crs = ccrs.AlbersEqualArea(central_latitude=39, central_longitude=-120)
    
basins_means_aea_gdf = basins_means_gdf.to_crs(cartopy_aea_crs)

In [ ]:
if region == 'hma':
    hillshade_regional_da = hillshade_da.rio.clip_box(minx=55, miny=10, maxx=115, maxy=55, crs="EPSG:4326").rio.reproject(cartopy_aea_crs).coarsen(x=2,y=2, boundary='trim').mean()
if region == 'wus':
    hillshade_regional_da = hillshade_da.rio.clip_box(minx=-170, miny=25, maxx=-100, maxy=80, crs="EPSG:4326").rio.reproject(cartopy_aea_crs).coarsen(x=2,y=2, boundary='trim').mean()
hillshade_regional_da

In [ ]:
# basins_means_wus_gdf = basins_means_aea_gdf.cx[-0.2E7:0.2E7, -0.35E7:0.35E7]
# basins_means_wus_gdf
basins_means_roi_gdf = basins_means_aea_gdf.cx[-0.2E7:0.2E7, -0.35E7:0.35E7]
basins_means_roi_gdf

In [ ]:
# percent of precipitation falling as snow and April-1 SWE per basin come from snow_water.ipynb
# (ERA5-Land via Earth Engine), written as a provenance-stamped results table
snow_water_csv = paths.resultsdir('river_basins', config.version) / 'river_basin_snow_water.csv'
if snow_water_csv.exists():
    snow_water = pd.read_csv(snow_water_csv)[['PFAF_ID', 'pct_precip_as_snow', 'swe_median']]
    basins_means_roi_gdf = basins_means_roi_gdf.drop(columns=[c for c in ('pct_precip_as_snow', 'swe_median') if c in basins_means_roi_gdf]).merge(snow_water, on='PFAF_ID', how='left')
else:
    print(f'{snow_water_csv} not found - run snow_water.ipynb first; the precipitation-as-snow panels will be empty')
    basins_means_roi_gdf['pct_precip_as_snow'] = np.nan
basins_means_roi_gdf[['PFAF_ID', 'runoff_onset_median', 'runoff_onset_mad', 'pct_precip_as_snow']].describe()

In [ ]:
f,axes=plt.subplots(nrows=2,ncols=5,figsize=(15,5.5),subplot_kw={'projection': cartopy_aea_crs},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(basins_ds.water_year.values, axes.flat):
    basins_means_aea_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2)
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)

    ax.set_title(f'WY{water_year}')
    # gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, linestyle='--', linewidth=0.5)
    # #gl.top_labels=False
    # # gl.bottom_labels=True
    # #gl.right_labels=False
    # # gl.left_labels=True

    # # turn on left labels for WY2015 and WY2020 only, turn on bottom labels for WY 2020-WY2024
    # if water_year == 2015 or water_year == 2020:
    #     gl.left_labels = True
    # else:
    #     gl.left_labels = False
    # if water_year in [2020, 2021, 2022, 2023, 2024]:
    #     gl.bottom_labels = True
    # else:
    #     gl.bottom_labels = False
        

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    #ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    #ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

# Add a single colorbar spanning both rows
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=-30, vmax=30)
sm = plt.cm.ScalarMappable(cmap='RdBu', norm=norm)
sm.set_array([])
cbar = f.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8, aspect=20, pad=0.02, extend='both')
cbar.set_label('Snowmelt runoff onset anomaly [days]')

#f.savefig(paths.figdir('river_basins', config.version) / 'hma_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
f.savefig(paths.figdir('river_basins', config.version) / f'{region}_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
#f.suptitle('Runoff Onset Anomaly by River Basin')
# f.tight_layout()

In [ ]:
# now create a two panel figure with the mean and mad runoff onset dates for the region
f,axs = plt.subplots(1,2,figsize=(12,4), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=cartopy_aea_crs,vmin=110,vmax=250,edgecolor='black', linewidth=0.2)
basins_means_aea_gdf.plot(ax=axs[1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=cartopy_aea_crs,vmin=0,vmax=30,edgecolor='black', linewidth=0.2)
for ax in axs:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())

axs[0].set_title("10-year Median Snowmelt Runoff Onset by River Basin")
axs[1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin")

In [ ]:
# now create a two panel figure with basin population and pct precip falling as snow

if region == 'hma':
    pop_max = 1E8
if region == 'wus':
    pop_max = 1E7
    
f,axs = plt.subplots(1,2,figsize=(10,4), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0],column='POPULATION', cmap='plasma', legend=True, legend_kwds={'label': "Basin Population"},transform=cartopy_aea_crs, norm=colors.LogNorm(vmin=100, vmax=pop_max),edgecolor='black', linewidth=0.2)
basins_means_roi_gdf.plot(ax=axs[1],column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': "%"},transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2,vmin=0,vmax=60)
for ax in axs:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    
axs[0].set_title("Basin Population")
axs[1].set_title("Basin Percentage of Precipitation Falling as Snow")

f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_population_and_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# major-rivers overlay: ESRI Data & Maps shapefile, not redistributable (see data/README.md) — skipped when absent;
# Natural Earth's ne_10m_rivers_lake_centerlines (public domain) is the drop-in if it is ever swapped
rivers_shp = paths.GEOMETRIES / 'majorrivers_0_0' / 'MajorRivers.shp'
if rivers_shp.exists():
    important_rivers_gdf = gpd.read_file(rivers_shp)
    important_rivers_gdf = important_rivers_gdf.to_crs(basins_means_aea_gdf.crs).cx[-0.2E7:0.2E7, -0.35E7:0.2E7]
    # include Indus, Ganges, Brahmaputra, Yangtze, Mekong, Amu Darya, Syr Darya, Huang He
    important_rivers_gdf = important_rivers_gdf[important_rivers_gdf['NAME'].isin(['Indus', 'Ganges', 'Brahmaputra', 'Yangtze', 'Mekong', 'Amu Darya', 'Syr Darya', 'Huang He'])]
else:
    important_rivers_gdf = None
    print(f'{rivers_shp} not found - the rivers overlay is skipped')
important_rivers_gdf

In [ ]:
f,ax=plt.subplots(figsize=(10,5),subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
important_rivers_gdf.plot(ax=ax, column='NAME', linewidth=1, zorder=1, legend=True)
# now label the rivers clo
for idx, row in important_rivers_gdf.iterrows():
    centroid = row['geometry'].centroid
    ax.text(centroid.x, centroid.y, row['NAME'], fontsize=8, zorder=2)
ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
#ax.legend()

In [ ]:
# now make a 4x4 figure of median runoff onset, mad, population, and pct precip as snow for HMA region
f,axs = plt.subplots(2,2,figsize=(10,7), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0,0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=cartopy_aea_crs,vmin=110,vmax=250,edgecolor='black', linewidth=0.2, alpha=0.8)
basins_means_aea_gdf.plot(ax=axs[0,1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=cartopy_aea_crs,vmin=0,vmax=30,edgecolor='black', linewidth=0.2,alpha=0.8)
basins_means_aea_gdf.plot(ax=axs[1,0],column='POPULATION', cmap='plasma', legend=True, legend_kwds={'label': "Basin Population"},transform=cartopy_aea_crs, norm=colors.LogNorm(vmin=100, vmax=1E8),edgecolor='black', linewidth=0.2, alpha=0.8)
basins_means_roi_gdf.plot(ax=axs[1,1],column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': "%"},transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2,vmin=0,vmax=60, alpha=0.8)

for ax in axs.flat:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    # gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True
    
    if important_rivers_gdf is not None:
        important_rivers_gdf.plot(ax=ax, color='black', linewidth=1, zorder=2)

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    #ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    #ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    
axs[0,0].set_title("10-year Median Snowmelt Runoff Onset")
axs[0,1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset")
axs[1,0].set_title("Basin Population")
axs[1,1].set_title("Basin Percentage of Precipitation Falling as Snow")

f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_runoff_onset_population_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)